In [1]:
import numpy as np
import pandas as pd

from linearmodels import PooledOLS          # Pooled model
from linearmodels import RandomEffects      # Random-effect model
from linearmodels import PanelOLS           # Fixed-effect model
from linearmodels import FirstDifferenceOLS # First difference model

from linearmodels.panel import compare      # Compare the results of multiple models
from statsmodels.api import add_constant    # for matrices of regression design

# 1.1 Модель 1. Результаты оценивания

In [2]:
df = pd.read_csv('Wages.csv')
df.head()

,exp,wks,bluecol,ind,south,smsa,married,sex,union,ed,black,lwage,id,time
0,3,32,no,0,yes,no,yes,male,no,9,no,5.56068,1,1
1,4,43,no,0,yes,no,yes,male,no,9,no,5.72031,1,2
2,5,40,no,0,yes,no,yes,male,no,9,no,5.99645,1,3
3,6,39,no,0,yes,no,yes,male,no,9,no,5.99645,1,4
4,7,42,no,1,yes,no,yes,male,no,9,no,6.06146,1,5


In [3]:
panel_df = df.set_index(['id', 'time'])
panel_df.head()

exp  wks bluecol  ind south smsa married   sex union  ed black  \
id time                                                                   
1  1       3   32      no    0   yes   no     yes  male    no   9    no   
   2       4   43      no    0   yes   no     yes  male    no   9    no   
   3       5   40      no    0   yes   no     yes  male    no   9    no   
   4       6   39      no    0   yes   no     yes  male    no   9    no   
   5       7   42      no    1   yes   no     yes  male    no   9    no   

           lwage  
id time           
1  1     5.56068  
   2     5.72031  
   3     5.99645  
   4     5.99645  
   5     6.06146

In [4]:
panel_df['d_lwage'] = panel_df['lwage'].groupby(level=0).diff()
panel_df.head(20)

exp  wks bluecol  ind south smsa married   sex union  ed black  \
id time                                                                   
1  1       3   32      no    0   yes   no     yes  male    no   9    no   
   2       4   43      no    0   yes   no     yes  male    no   9    no   
   3       5   40      no    0   yes   no     yes  male    no   9    no   
   4       6   39      no    0   yes   no     yes  male    no   9    no   
   5       7   42      no    1   yes   no     yes  male    no   9    no   
   6       8   35      no    1   yes   no     yes  male    no   9    no   
   7       9   32      no    1   yes   no     yes  male    no   9    no   
2  1      30   34     yes    0    no   no     yes  male    no  11    no   
   2      31   27     yes    0    no   no     yes  male    no  11    no   
   3      32   33     yes    1    no   no     yes  male   yes  11    no   
   4      33   30     yes    1    no   no     yes  male    no  11    no   
   5      34   30     yes    1    no   no     yes  male    no  11    no   
   6      35   37     yes    1    no   no     yes  male    no  11    no   
   7      36   30     yes    1    no   no     yes  male    no  11    no   
3  1       6   50     yes    1    no   no     yes  male   yes  12    no   
   2       7   51     yes    1    no   no     yes  male   yes  12    no   
   3       8   50     yes    1    no   no     yes  male   yes  12    no   
   4       9   52     yes    1    no   no     yes  male   yes  12    no   
   5      10   52     yes    1    no   no     yes  male   yes  12    no   
   6      11   52     yes    1    no   no      no  male   yes  12    no   

           lwage  d_lwage  
id time                    
1  1     5.56068      NaN  
   2     5.72031  0.15963  
   3     5.99645  0.27614  
   4     5.99645  0.00000  
   5     6.06146  0.06501  
   6     6.17379  0.11233  
   7     6.24417  0.07038  
2  1     6.16331      NaN  
   2     6.21461  0.05130  
   3     6.26340  0.04879  
   4     6.54391  0.28051  
   5     6.69703  0.15312  
   6     6.79122  0.09419  
   7     6.81564  0.02442  
3  1     5.65249      NaN  
   2     6.43615  0.78366  
   3     6.54822  0.11207  
   4     6.60259  0.05437  
   5     6.69580  0.09321  
   6     6.77878  0.08298

In [5]:
mod_pl = PooledOLS.from_formula(formula='d_lwage~1+ed+exp+I(exp**2)+wks', data=panel_df)
res_pl = mod_pl.fit()
res_pl.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:921: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept      0.0904
ed             0.0020
exp           -0.0026
I(exp ** 2)    0.0000
wks            0.0003
Name: parameter, dtype: float64

In [6]:
mod_re = RandomEffects.from_formula(formula='d_lwage~1+ed+exp+I(exp**2)+wks', data=panel_df)
res_re = mod_re.fit()
res_re.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept      0.0904
ed             0.0020
exp           -0.0026
I(exp ** 2)    0.0000
wks            0.0003
Name: parameter, dtype: float64

In [7]:
mod_fe = PanelOLS.from_formula(formula='d_lwage~1+ed+exp+I(exp**2)+wks+EntityEffects', data=panel_df, drop_absorbed=True)
res_fe = mod_fe.fit()
res_fe.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
C:\Users\stud.LIMM\AppData\Local\Temp\ipykernel_6676\3154518583.py:2: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

ed

  res_fe = mod_fe.fit()


Intercept      0.2480
exp           -0.0101
I(exp ** 2)    0.0001
wks           -0.0002
Name: parameter, dtype: float64

# 2.1 Модель 1. Результаты оценивания

In [8]:
df = pd.read_csv('Cigar.csv')
df.head()

,state,year,price,pop,pop16,cpi,ndi,sales,pimin
0,1,63,28.6,3383.0,2236.5,30.6,1558.304530,93.9,26.1
1,1,64,29.8,3431.0,2276.7,31.0,1684.073202,95.4,27.5
2,1,65,29.8,3486.0,2327.5,31.5,1809.841875,98.5,28.9
3,1,66,31.5,3524.0,2369.7,32.4,1915.160357,96.4,29.5
4,1,67,31.6,3533.0,2393.7,33.4,2023.546368,95.5,29.6


In [9]:
panel_df = df.set_index(['state', 'year'])
panel_df.head(20)

price     pop   pop16   cpi          ndi  sales  pimin
state year                                                        
1     63     28.6  3383.0  2236.5  30.6  1558.304530   93.9   26.1
      64     29.8  3431.0  2276.7  31.0  1684.073202   95.4   27.5
      65     29.8  3486.0  2327.5  31.5  1809.841875   98.5   28.9
      66     31.5  3524.0  2369.7  32.4  1915.160357   96.4   29.5
      67     31.6  3533.0  2393.7  33.4  2023.546368   95.5   29.6
      68     35.6  3522.0  2405.2  34.8  2202.485536   88.4   32.0
      69     36.6  3531.0  2411.9  36.7  2377.334666   90.1   32.8
      70     39.6  3444.0  2394.6  38.8  2591.039159   89.8   34.3
      71     42.7  3481.0  2443.5  40.5  2785.315971   95.4   35.8
      72     42.3  3511.0  2484.7  41.8  3034.808297  101.1   37.4
      73     42.1  3540.0  2526.0  44.4  3387.574086  102.9   37.3
      74     43.1  3574.0  2573.9  49.3  3718.867175  108.2   41.4
      75     46.6  3614.0  2623.7  53.8  4087.993117  111.7   43.0
      76     50.4  3657.0  2677.4  56.9  4486.771835  116.2   46.4
      77     50.1  3690.0  2719.6  60.6  4899.865687  117.1   48.8
      78     55.1  3728.0  2764.6  65.2  5450.998326  123.0   53.6
      79     56.8  3769.0  2810.7  72.6  5957.140545  121.4   56.5
      80     60.6  3894.0  2898.9  82.4  6466.350293  123.2   59.3
      81     68.8  3917.0  2924.7  90.9  7042.023161  119.6   62.6
      82     73.1  3943.0  2953.5  96.5  7505.219979  119.1   67.8

In [10]:
'd_'+panel_df.columns.values

array(['d_price', 'd_pop', 'd_pop16', 'd_cpi', 'd_ndi', 'd_sales',
       'd_pimin'], dtype=object)

In [11]:
panel_df['d_'+panel_df.columns.values] = panel_df.groupby(level=0).diff()
panel_df.head(20)

price     pop   pop16   cpi          ndi  sales  pimin  d_price  \
state year                                                                    
1     63     28.6  3383.0  2236.5  30.6  1558.304530   93.9   26.1      NaN   
      64     29.8  3431.0  2276.7  31.0  1684.073202   95.4   27.5      1.2   
      65     29.8  3486.0  2327.5  31.5  1809.841875   98.5   28.9      0.0   
      66     31.5  3524.0  2369.7  32.4  1915.160357   96.4   29.5      1.7   
      67     31.6  3533.0  2393.7  33.4  2023.546368   95.5   29.6      0.1   
      68     35.6  3522.0  2405.2  34.8  2202.485536   88.4   32.0      4.0   
      69     36.6  3531.0  2411.9  36.7  2377.334666   90.1   32.8      1.0   
      70     39.6  3444.0  2394.6  38.8  2591.039159   89.8   34.3      3.0   
      71     42.7  3481.0  2443.5  40.5  2785.315971   95.4   35.8      3.1   
      72     42.3  3511.0  2484.7  41.8  3034.808297  101.1   37.4     -0.4   
      73     42.1  3540.0  2526.0  44.4  3387.574086  102.9   37.3     -0.2   
      74     43.1  3574.0  2573.9  49.3  3718.867175  108.2   41.4      1.0   
      75     46.6  3614.0  2623.7  53.8  4087.993117  111.7   43.0      3.5   
      76     50.4  3657.0  2677.4  56.9  4486.771835  116.2   46.4      3.8   
      77     50.1  3690.0  2719.6  60.6  4899.865687  117.1   48.8     -0.3   
      78     55.1  3728.0  2764.6  65.2  5450.998326  123.0   53.6      5.0   
      79     56.8  3769.0  2810.7  72.6  5957.140545  121.4   56.5      1.7   
      80     60.6  3894.0  2898.9  82.4  6466.350293  123.2   59.3      3.8   
      81     68.8  3917.0  2924.7  90.9  7042.023161  119.6   62.6      8.2   
      82     73.1  3943.0  2953.5  96.5  7505.219979  119.1   67.8      4.3   

            d_pop  d_pop16  d_cpi       d_ndi  d_sales  d_pimin  
state year                                                       
1     63      NaN      NaN    NaN         NaN      NaN      NaN  
      64     48.0     40.2    0.4  125.768673      1.5      1.4  
      65     55.0     50.8    0.5  125.768673      3.1      1.4  
      66     38.0     42.2    0.9  105.318482     -2.1      0.6  
      67      9.0     24.0    1.0  108.386011     -0.9      0.1  
      68    -11.0     11.5    1.4  178.939168     -7.1      2.4  
      69      9.0      6.7    1.9  174.849130      1.7      0.8  
      70    -87.0    -17.3    2.1  213.704493     -0.3      1.5  
      71     37.0     48.9    1.7  194.276812      5.6      1.5  
      72     30.0     41.2    1.3  249.492326      5.7      1.6  
      73     29.0     41.3    2.6  352.765789      1.8     -0.1  
      74     34.0     47.9    4.9  331.293089      5.3      4.1  
      75     40.0     49.8    4.5  369.125942      3.5      1.6  
      76     43.0     53.7    3.1  398.778718      4.5      3.4  
      77     33.0     42.2    3.7  413.093852      0.9      2.4  
      78     38.0     45.0    4.6  551.132639      5.9      4.8  
      79     41.0     46.1    7.4  506.142219     -1.6      2.9  
      80    125.0     88.2    9.8  509.209748      1.8      2.8  
      81     23.0     25.8    8.5  575.672868     -3.6      3.3  
      82     26.0     28.8    5.6  463.196819     -0.5      5.2

In [12]:
mod_pl = PooledOLS.from_formula(formula='d_sales~1+d_price+d_pop+d_pop16+d_cpi+d_ndi+d_pimin', data=panel_df)
res_pl = mod_pl.fit()
res_pl.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:921: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept    0.4837
d_price     -0.3354
d_pop        0.0003
d_pop16      0.0002
d_cpi       -0.0799
d_ndi        0.0014
d_pimin     -0.0384
Name: parameter, dtype: float64

In [13]:
mod_re = RandomEffects.from_formula(formula='d_sales~1+d_price+d_pop+d_pop16+d_cpi+d_ndi+d_pimin', data=panel_df)
res_re = mod_re.fit()
res_re.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept    0.4837
d_price     -0.3354
d_pop        0.0003
d_pop16      0.0002
d_cpi       -0.0799
d_ndi        0.0014
d_pimin     -0.0384
Name: parameter, dtype: float64

In [14]:
mod_fe = PanelOLS.from_formula(formula='d_sales~1+d_price+d_pop+d_pop16+d_cpi+d_ndi+d_pimin+EntityEffects', data=panel_df, drop_absorbed=True)
res_fe = mod_fe.fit()
res_fe.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept    0.1891
d_price     -0.3295
d_pop        0.0047
d_pop16      0.0002
d_cpi       -0.1608
d_ndi        0.0024
d_pimin     -0.0692
Name: parameter, dtype: float64

# 2.2 Модель 2. Результаты оценивания

In [15]:
df = pd.read_csv('Cigar.csv')
df.head()

,state,year,price,pop,pop16,cpi,ndi,sales,pimin
0,1,63,28.6,3383.0,2236.5,30.6,1558.304530,93.9,26.1
1,1,64,29.8,3431.0,2276.7,31.0,1684.073202,95.4,27.5
2,1,65,29.8,3486.0,2327.5,31.5,1809.841875,98.5,28.9
3,1,66,31.5,3524.0,2369.7,32.4,1915.160357,96.4,29.5
4,1,67,31.6,3533.0,2393.7,33.4,2023.546368,95.5,29.6


In [16]:
panel_df = df.set_index(['state', 'year'])
panel_df.head(20)

price     pop   pop16   cpi          ndi  sales  pimin
state year                                                        
1     63     28.6  3383.0  2236.5  30.6  1558.304530   93.9   26.1
      64     29.8  3431.0  2276.7  31.0  1684.073202   95.4   27.5
      65     29.8  3486.0  2327.5  31.5  1809.841875   98.5   28.9
      66     31.5  3524.0  2369.7  32.4  1915.160357   96.4   29.5
      67     31.6  3533.0  2393.7  33.4  2023.546368   95.5   29.6
      68     35.6  3522.0  2405.2  34.8  2202.485536   88.4   32.0
      69     36.6  3531.0  2411.9  36.7  2377.334666   90.1   32.8
      70     39.6  3444.0  2394.6  38.8  2591.039159   89.8   34.3
      71     42.7  3481.0  2443.5  40.5  2785.315971   95.4   35.8
      72     42.3  3511.0  2484.7  41.8  3034.808297  101.1   37.4
      73     42.1  3540.0  2526.0  44.4  3387.574086  102.9   37.3
      74     43.1  3574.0  2573.9  49.3  3718.867175  108.2   41.4
      75     46.6  3614.0  2623.7  53.8  4087.993117  111.7   43.0
      76     50.4  3657.0  2677.4  56.9  4486.771835  116.2   46.4
      77     50.1  3690.0  2719.6  60.6  4899.865687  117.1   48.8
      78     55.1  3728.0  2764.6  65.2  5450.998326  123.0   53.6
      79     56.8  3769.0  2810.7  72.6  5957.140545  121.4   56.5
      80     60.6  3894.0  2898.9  82.4  6466.350293  123.2   59.3
      81     68.8  3917.0  2924.7  90.9  7042.023161  119.6   62.6
      82     73.1  3943.0  2953.5  96.5  7505.219979  119.1   67.8

In [19]:
var_names=np.array(['price', 'pop', 'cpi'])

In [20]:
panel_df['lag_'+var_names] = panel_df[var_names].groupby(level=0).shift()
panel_df.head(20)

price     pop   pop16   cpi          ndi  sales  pimin  lag_price  \
state year                                                                      
1     63     28.6  3383.0  2236.5  30.6  1558.304530   93.9   26.1        NaN   
      64     29.8  3431.0  2276.7  31.0  1684.073202   95.4   27.5       28.6   
      65     29.8  3486.0  2327.5  31.5  1809.841875   98.5   28.9       29.8   
      66     31.5  3524.0  2369.7  32.4  1915.160357   96.4   29.5       29.8   
      67     31.6  3533.0  2393.7  33.4  2023.546368   95.5   29.6       31.5   
      68     35.6  3522.0  2405.2  34.8  2202.485536   88.4   32.0       31.6   
      69     36.6  3531.0  2411.9  36.7  2377.334666   90.1   32.8       35.6   
      70     39.6  3444.0  2394.6  38.8  2591.039159   89.8   34.3       36.6   
      71     42.7  3481.0  2443.5  40.5  2785.315971   95.4   35.8       39.6   
      72     42.3  3511.0  2484.7  41.8  3034.808297  101.1   37.4       42.7   
      73     42.1  3540.0  2526.0  44.4  3387.574086  102.9   37.3       42.3   
      74     43.1  3574.0  2573.9  49.3  3718.867175  108.2   41.4       42.1   
      75     46.6  3614.0  2623.7  53.8  4087.993117  111.7   43.0       43.1   
      76     50.4  3657.0  2677.4  56.9  4486.771835  116.2   46.4       46.6   
      77     50.1  3690.0  2719.6  60.6  4899.865687  117.1   48.8       50.4   
      78     55.1  3728.0  2764.6  65.2  5450.998326  123.0   53.6       50.1   
      79     56.8  3769.0  2810.7  72.6  5957.140545  121.4   56.5       55.1   
      80     60.6  3894.0  2898.9  82.4  6466.350293  123.2   59.3       56.8   
      81     68.8  3917.0  2924.7  90.9  7042.023161  119.6   62.6       60.6   
      82     73.1  3943.0  2953.5  96.5  7505.219979  119.1   67.8       68.8   

            lag_pop  lag_cpi  
state year                    
1     63        NaN      NaN  
      64     3383.0     30.6  
      65     3431.0     31.0  
      66     3486.0     31.5  
      67     3524.0     32.4  
      68     3533.0     33.4  
      69     3522.0     34.8  
      70     3531.0     36.7  
      71     3444.0     38.8  
      72     3481.0     40.5  
      73     3511.0     41.8  
      74     3540.0     44.4  
      75     3574.0     49.3  
      76     3614.0     53.8  
      77     3657.0     56.9  
      78     3690.0     60.6  
      79     3728.0     65.2  
      80     3769.0     72.6  
      81     3894.0     82.4  
      82     3917.0     90.9

In [22]:
mod_pl = PooledOLS.from_formula(formula='sales~1+price+lag_price+pop+lag_pop+pop16+cpi+lag_cpi+ndi+pimin', data=panel_df)
res_pl = mod_pl.fit()
res_pl.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:921: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept    140.1937
price         -0.9110
lag_price     -0.7451
pop           -0.0018
lag_pop       -0.0014
pop16          0.0032
cpi            0.0060
lag_cpi        0.1686
ndi            0.0060
pimin          0.6367
Name: parameter, dtype: float64

In [23]:
mod_re = RandomEffects.from_formula(formula='sales~1+price+lag_price+pop+lag_pop+pop16+cpi+lag_cpi+ndi+pimin', data=panel_df)
res_re = mod_re.fit()
res_re.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept    128.0012
price         -0.3856
lag_price     -0.2992
pop            0.0087
lag_pop       -0.0152
pop16          0.0082
cpi            1.5026
lag_cpi       -0.7010
ndi           -0.0049
pimin          0.2819
Name: parameter, dtype: float64

In [24]:
mod_fe = PanelOLS.from_formula(formula='sales~1+price+lag_price+pop+lag_pop+pop16+cpi+lag_cpi+ndi+pimin+EntityEffects', data=panel_df, drop_absorbed=True)
res_fe = mod_fe.fit()
res_fe.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept    127.8755
price         -0.3733
lag_price     -0.2901
pop            0.0093
lag_pop       -0.0160
pop16          0.0085
cpi            1.5442
lag_cpi       -0.7189
ndi           -0.0052
pimin          0.2782
Name: parameter, dtype: float64

# 3.2 Модель 2. Результаты оценивания

In [25]:
df = pd.read_csv('EmplUK.csv')
df.head()

,firm,year,sector,emp,wage,capital,output
0,1,1977,7,5.041,13.1516,0.5894,95.707199
1,1,1978,7,5.600,12.3018,0.6318,97.356903
2,1,1979,7,5.015,12.8395,0.6771,99.608299
3,1,1980,7,4.715,13.8039,0.6171,100.550100
4,1,1981,7,4.093,14.2897,0.5076,99.558098


In [26]:
panel_df = df.set_index(['firm', 'year'])
panel_df.head(20)

sector        emp       wage    capital      output
firm year                                                     
1    1977       7   5.041000  13.151600   0.589400   95.707199
     1978       7   5.600000  12.301800   0.631800   97.356903
     1979       7   5.015000  12.839500   0.677100   99.608299
     1980       7   4.715000  13.803900   0.617100  100.550100
     1981       7   4.093000  14.289700   0.507600   99.558098
     1982       7   3.166000  14.868100   0.422900   98.615097
     1983       7   2.936000  13.778400   0.392000  100.030100
2    1977       7  71.319000  14.790900  16.936300   95.707199
     1978       7  70.642998  14.103600  17.242201   97.356903
     1979       7  70.917999  14.953400  17.541300   99.608299
     1980       7  72.030998  15.491000  17.657400  100.550100
     1981       7  73.689003  16.196899  16.713301   99.558098
     1982       7  72.418999  16.131399  16.246901   98.615097
     1983       7  68.517998  16.305099  17.369600  100.030100
3    1977       7  19.156000  22.691999   7.097500   95.707199
     1978       7  19.440001  20.693800   6.946900   97.356903
     1979       7  19.900000  21.204800   6.856500   99.608299
     1980       7  20.240000  22.197001   6.654700  100.550100
     1981       7  19.570000  24.871401   6.213600   99.558098
     1982       7  18.125000  24.844700   5.714600   98.615097

In [27]:
var_names=np.array(['emp', 'wage', 'capital', 'output'])

In [28]:
panel_df['d_log_'+var_names] = np.log(panel_df[var_names]).groupby(level=0).diff()
panel_df.head(20)

sector        emp       wage    capital      output  d_log_emp  \
firm year                                                                   
1    1977       7   5.041000  13.151600   0.589400   95.707199        NaN   
     1978       7   5.600000  12.301800   0.631800   97.356903   0.105162   
     1979       7   5.015000  12.839500   0.677100   99.608299  -0.110333   
     1980       7   4.715000  13.803900   0.617100  100.550100  -0.061684   
     1981       7   4.093000  14.289700   0.507600   99.558098  -0.141471   
     1982       7   3.166000  14.868100   0.422900   98.615097  -0.256809   
     1983       7   2.936000  13.778400   0.392000  100.030100  -0.075421   
2    1977       7  71.319000  14.790900  16.936300   95.707199        NaN   
     1978       7  70.642998  14.103600  17.242201   97.356903  -0.009524   
     1979       7  70.917999  14.953400  17.541300   99.608299   0.003885   
     1980       7  72.030998  15.491000  17.657400  100.550100   0.015572   
     1981       7  73.689003  16.196899  16.713301   99.558098   0.022757   
     1982       7  72.418999  16.131399  16.246901   98.615097  -0.017385   
     1983       7  68.517998  16.305099  17.369600  100.030100  -0.055372   
3    1977       7  19.156000  22.691999   7.097500   95.707199        NaN   
     1978       7  19.440001  20.693800   6.946900   97.356903   0.014717   
     1979       7  19.900000  21.204800   6.856500   99.608299   0.023387   
     1980       7  20.240000  22.197001   6.654700  100.550100   0.016941   
     1981       7  19.570000  24.871401   6.213600   99.558098  -0.033663   
     1982       7  18.125000  24.844700   5.714600   98.615097  -0.076706   

           d_log_wage  d_log_capital  d_log_output  
firm year                                           
1    1977         NaN            NaN           NaN  
     1978   -0.066798       0.069468      0.017090  
     1979    0.042781       0.069246      0.022862  
     1980    0.072425      -0.092788      0.009411  
     1981    0.034588      -0.195337     -0.009915  
     1982    0.039679      -0.182558     -0.009517  
     1983   -0.076116      -0.075874      0.014247  
2    1977         NaN            NaN           NaN  
     1978   -0.047582       0.017901      0.017090  
     1979    0.058509       0.017198      0.022862  
     1980    0.035321       0.006597      0.009411  
     1981    0.044561      -0.054950     -0.009915  
     1982   -0.004052      -0.028303     -0.009517  
     1983    0.010710       0.066819      0.014247  
3    1977         NaN            NaN           NaN  
     1978   -0.092178      -0.021447      0.017090  
     1979    0.024393      -0.013098      0.022862  
     1980    0.045730      -0.029874      0.009411  
     1981    0.113761      -0.068583     -0.009915  
     1982   -0.001074      -0.083716     -0.009517

In [29]:
mod_pl = PooledOLS.from_formula(formula='d_log_emp~1+d_log_wage+d_log_capital+d_log_output', data=panel_df)
res_pl = mod_pl.fit()
res_pl.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:921: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept       -0.0180
d_log_wage      -0.4160
d_log_capital    0.4083
d_log_output     0.4090
Name: parameter, dtype: float64

In [30]:
mod_re = RandomEffects.from_formula(formula='d_log_emp~1+d_log_wage+d_log_capital+d_log_output', data=panel_df)
res_re = mod_re.fit()
res_re.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:2759: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept       -0.0180
d_log_wage      -0.4160
d_log_capital    0.4083
d_log_output     0.4090
Name: parameter, dtype: float64

In [31]:
mod_fe = PanelOLS.from_formula(formula='d_log_emp~1+d_log_wage+d_log_capital+d_log_output+EntityEffects', data=panel_df, drop_absorbed=True)
res_fe = mod_fe.fit()
res_fe.params.round(4)

C:\ProgramData\anaconda3\envs\courses\Lib\site-packages\linearmodels\panel\model.py:1260: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept       -0.0188
d_log_wage      -0.4745
d_log_capital    0.3523
d_log_output     0.4510
Name: parameter, dtype: float64